In [ ]:
# =============================================================================
# BERT预训练模型完整实现
# =============================================================================
# BERT (Bidirectional Encoder Representations from Transformers) 是一种双向Transformer编码器预训练模型
# 主要包含两个预训练任务：
#   1. MLM (Masked Language Model): 掩蔽语言模型 - 预测被掩蔽的词
#   2. NSP (Next Sentence Prediction): 下一句预测 - 判断两个句子是否连续
#
# 本文档实现：
#   - BERT编码器（基于Transformer Encoder）
#   - 输入处理（token embedding + segment embedding + position embedding）
#   - MLM任务头部
#   - NSP任务头部
#   - 完整BERT模型组装
# =============================================================================

import torch
from torch import nn
from d2l import torch as d2l

#@save
def get_tokens_and_segments(tokens_a, tokens_b=None):
    """
    获取输入序列的词元及其片段索引
    
    BERT的输入格式：[<cls>] + tokens_a + [<sep>] + tokens_b + [<sep>]
    
    参数:
        tokens_a: 第一个句子的词元列表
        tokens_b: 第二个句子的词元列表（可选，用于NSP任务）
    
    返回:
        tokens: 完整的词元序列
        segments: 片段索引（0表示属于句子A，1表示属于句子B）
    
    示例:
        >>> tokens_a = ['hello', 'world']
        >>> tokens_b = ['how', 'are', 'you']
        >>> get_tokens_and_segments(tokens_a, tokens_b)
        (['<cls>', 'hello', 'world', '<sep>', 'how', 'are', 'you', '<sep>'],
         [0, 0, 0, 0, 1, 1, 1, 1])
    """
    # 构建词元序列：以<cls>开头，句子A后跟<sep>
    tokens = ['<cls>'] + tokens_a + ['<sep>']
    # segments: 0和1分别标记片段A和B
    # <cls>和句子A、<sep>都属于片段0
    segments = [0] * (len(tokens_a) + 2)
    
    if tokens_b is not None:
        # 如果有第二个句子，添加tokens_b和其结束标记<sep>
        tokens += tokens_b + ['<sep>']
        # tokens_b的词元都属于片段1
        segments += [1] * (len(tokens_b) + 1)
    
    return tokens, segments


#@save
class BERTEncoder(nn.Module):
    """
    BERT编码器 - 基于Transformer Encoder架构
    
    BERT的核心思想：通过双向注意力机制，让每个词都能关注到句子中所有其他词
    这与GPT（单向）和ELMo（浅层双向）不同，BERT是真正的深层双向表示
    
    架构组成：
    1. Token Embedding: 将词元映射为向量
    2. Segment Embedding: 区分句子A和句子B
    3. Position Embedding: 位置信息（可学习的，不是正弦编码）
    4. Transformer Encoder Blocks: 多层自注意力 + 前馈网络
    """
    
    def __init__(self, vocab_size, num_hiddens, norm_shape, ffn_num_input,
                 ffn_num_hiddens, num_heads, num_layers, dropout,
                 max_len=1000, key_size=768, query_size=768, value_size=768,
                 **kwargs):
        """
        参数:
            vocab_size: 词汇表大小
            num_hiddens: 隐藏层维度（即embedding维度，通常768）
            norm_shape: 层归一化的形状（通常是[num_hiddens]）
            ffn_num_input: 前馈网络输入维度
            ffn_num_hiddens: 前馈网络隐藏层维度（通常是num_hiddens的4倍）
            num_heads: 多头注意力头数
            num_layers: Transformer编码器层数
            dropout: dropout概率
            max_len: 最大序列长度（用于位置嵌入）
            key_size, query_size, value_size: 注意力机制的维度
        """
        super(BERTEncoder, self).__init__(**kwargs)
        
        # ===== 嵌入层 =====
        # Token Embedding: 将词元ID映射为密集向量
        # 形状变化: (batch_size, seq_len) -> (batch_size, seq_len, num_hiddens)
        self.token_embedding = nn.Embedding(vocab_size, num_hiddens)
        
        # Segment Embedding: 区分两个句子（A和B）
        # 只有两个可能的值（0和1），所以num_embeddings=2
        self.segment_embedding = nn.Embedding(2, num_hiddens)
        
        # ===== Transformer编码器块 =====
        # 堆叠多个EncoderBlock，每个包含：多头注意力 + 前馈网络 + 残差连接 + 层归一化
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module(f"{i}", d2l.EncoderBlock(
                key_size, query_size, value_size, num_hiddens, norm_shape,
                ffn_num_input, ffn_num_hiddens, num_heads, dropout, True))
        
        # ===== 位置嵌入 =====
        # BERT使用的是可学习的位置嵌入，而非Transformer原论文中的固定正弦编码
        # 形状: (1, max_len, num_hiddens)
        # 第一个维度为1是为了广播到所有样本
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, num_hiddens))

    def forward(self, tokens, segments, valid_lens):
        """
        前向传播
        
        参数:
            tokens: 词元ID，形状 (batch_size, seq_len)
            segments: 片段索引，形状 (batch_size, seq_len)
            valid_lens: 有效长度，用于注意力掩码，形状 (batch_size,)
        
        返回:
            X: 编码后的表示，形状 (batch_size, seq_len, num_hiddens)
        
        维度变化:
            输入: (batch_size, seq_len)
            Token Embedding: (batch_size, seq_len, num_hiddens)
            + Segment Embedding: (batch_size, seq_len, num_hiddens)
            + Position Embedding: (batch_size, seq_len, num_hiddens)
            经过Transformer Blocks: (batch_size, seq_len, num_hiddens)
        """
        # 在以下代码段中，X的形状保持不变：(batch_size, seq_len, num_hiddens)
        
        # 1. Token Embedding: 将词元ID转为向量
        X = self.token_embedding(tokens)
        
        # 2. Segment Embedding: 添加句子片段信息
        X = X + self.segment_embedding(segments)
        
        # 3. Position Embedding: 添加位置信息
        # 只取前seq_len个位置嵌入，避免浪费计算
        X = X + self.pos_embedding.data[:, :X.shape[1], :]
        
        # 4. 通过Transformer编码器块
        for blk in self.blks:
            X = blk(X, valid_lens)
        
        return X


# =============================================================================
# BERT编码器测试
# =============================================================================
# 定义模型超参数
vocab_size, num_hiddens, ffn_num_hiddens, num_heads = 10000, 768, 1024, 4
norm_shape, ffn_num_input, num_layers, dropout = [768], 768, 2, 0.2

# 创建BERT编码器实例
encoder = BERTEncoder(vocab_size, num_hiddens, norm_shape, ffn_num_input,
                      ffn_num_hiddens, num_heads, num_layers, dropout)

# 生成随机测试数据
# tokens: 2个样本，每个样本8个词元
tokens = torch.randint(0, vocab_size, (2, 8))

# segments: 第1个样本前4个是句子A，后4个是句子B
#          第2个样本前3个是句子A，后5个是句子B
segments = torch.tensor([[0, 0, 0, 0, 1, 1, 1, 1], 
                         [0, 0, 0, 1, 1, 1, 1, 1]])

# 前向传播
encoded_X = encoder(tokens, segments, None)

# 验证输出形状
print(f"输入tokens形状: {tokens.shape}")
print(f"输入segments形状: {segments.shape}")
print(f"输出encoded_X形状: {encoded_X.shape}")
# 预期输出: torch.Size([2, 8, 768]) - (batch_size, seq_len, num_hiddens)


#@save
class MaskLM(nn.Module):
    """
    BERT的掩蔽语言模型任务（Masked Language Model）
    
    MLM任务原理：
    1. 随机选择输入序列中约15%的词元
    2. 对这些词元进行三种处理：
       - 80%的概率替换为<mask>标记
       - 10%的概率保持原词
       - 10%的概率替换为随机词
    3. 模型预测这些被掩蔽位置的原词
    
    为什么要这样做？
    - 迫使模型基于双向上下文来理解词义
    - 避免模型只是简单地"记住"词的位置关系
    """
    
    def __init__(self, vocab_size, num_hiddens, num_inputs=768, **kwargs):
        """
        参数:
            vocab_size: 词汇表大小
            num_hiddens: MLP隐藏层维度
            num_inputs: 输入维度（等于BERT的num_hiddens）
        """
        super(MaskLM, self).__init__(**kwargs)
        
        # MLP结构：Linear -> ReLU -> LayerNorm -> Linear
        # 输入：被掩蔽位置的表示 (batch_size, num_mlm_preds, num_inputs)
        # 输出：每个位置的词汇表分布 (batch_size, num_mlm_preds, vocab_size)
        self.mlp = nn.Sequential(
            nn.Linear(num_inputs, num_hiddens),  # 降维/升维
            nn.ReLU(),                           # 非线性激活
            nn.LayerNorm(num_hiddens),           # 层归一化，稳定训练
            nn.Linear(num_hiddens, vocab_size)   # 映射到词汇表
        )

    def forward(self, X, pred_positions):
        """
        前向传播 - 预测被掩蔽位置的词
        
        参数:
            X: BERT编码器的输出，形状 (batch_size, seq_len, num_hiddens)
            pred_positions: 需要预测的位置，形状 (batch_size, num_mlm_preds)
                           表示每个样本中被掩蔽的位置索引
        
        返回:
            mlm_Y_hat: 预测结果，形状 (batch_size, num_mlm_preds, vocab_size)
        
        关键操作：使用高级索引提取被掩蔽位置的表示
        """
        num_pred_positions = pred_positions.shape[1]  # 每个样本要预测的位置数
        
        # 将pred_positions展平为一维，便于索引
        # 形状: (batch_size * num_mlm_preds,)
        pred_positions = pred_positions.reshape(-1)
        
        batch_size = X.shape[0]
        
        # 创建批次索引
        # 假设batch_size=2，num_pred_positions=3
        # batch_idx = [0, 0, 0, 1, 1, 1]
        # 这表示前3个预测位置属于第0个样本，后3个属于第1个样本
        batch_idx = torch.arange(0, batch_size)
        batch_idx = torch.repeat_interleave(batch_idx, num_pred_positions)
        
        # 高级索引：提取被掩蔽位置的表示
        # X[batch_idx, pred_positions] 的形状: (batch_size * num_mlm_preds, num_hiddens)
        # 举例：X[[0,0,0,1,1,1], [pos1,pos2,pos3,pos4,pos5,pos6]]
        masked_X = X[batch_idx, pred_positions]
        
        # 恢复形状：(batch_size, num_mlm_preds, num_hiddens)
        masked_X = masked_X.reshape((batch_size, num_pred_positions, -1))
        
        # 通过MLP预测词汇表分布
        mlm_Y_hat = self.mlp(masked_X)
        
        return mlm_Y_hat


# =============================================================================
# MLM任务测试
# =============================================================================
mlm = MaskLM(vocab_size, num_hiddens)

# 假设我们要预测的位置：每个样本预测3个位置
# 第1个样本预测位置1, 5, 2
# 第2个样本预测位置6, 1, 5
mlm_positions = torch.tensor([[1, 5, 2], [6, 1, 5]])

# 前向传播
mlm_Y_hat = mlm(encoded_X, mlm_positions)

print(f"\nMLM测试:")
print(f"预测位置形状: {mlm_positions.shape}")
print(f"MLM预测输出形状: {mlm_Y_hat.shape}")
# 预期: torch.Size([2, 3, 10000]) - (batch_size, num_mlm_preds, vocab_size)

# 模拟真实标签
mlm_Y = torch.tensor([[7, 8, 9], [10, 20, 30]])

# 计算损失
loss = nn.CrossEntropyLoss(reduction='none')
mlm_l = loss(mlm_Y_hat.reshape((-1, vocab_size)), mlm_Y.reshape(-1))

print(f"MLM损失形状: {mlm_l.shape}")
# 预期: torch.Size([6]) - (batch_size * num_mlm_preds,)


#@save
class NextSentencePred(nn.Module):
    """
    BERT的下一句预测任务（Next Sentence Prediction）
    
    NSP任务原理：
    - 输入两个句子（A和B），判断B是否是A的下一句
    - 正样本：从文档中连续采样的两个句子
    - 负样本：从不同文档中随机采样的句子对
    
    为什么需要NSP？
    - 学习句子级别的关系理解
    - 对下游任务（如问答、自然语言推理）很有帮助
    
    注意：后续研究（如RoBERTa）发现NSP任务效果有限，有时会去掉
    """
    
    def __init__(self, num_inputs, **kwargs):
        """
        参数:
            num_inputs: 输入维度（等于BERT的num_hiddens）
        """
        super(NextSentencePred, self).__init__(**kwargs)
        # 简单的线性分类器：输入<cls>标记的表示，输出二分类结果
        self.output = nn.Linear(num_inputs, 2)

    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入表示，形状 (batch_size, num_hiddens)
               通常是BERT输出中<cls>标记对应的表示
        
        返回:
            二分类 logits，形状 (batch_size, 2)
        """
        return self.output(X)


# =============================================================================
# NSP任务测试
# =============================================================================
# 将编码器输出展平，模拟提取<cls>标记的表示
# encoded_X形状: (2, 8, 768)
# 展平后: (2, 8*768) = (2, 6144)
encoded_X = torch.flatten(encoded_X, start_dim=1)

# 创建NSP预测器
nsp = NextSentencePred(encoded_X.shape[-1])

# 前向传播
nsp_Y_hat = nsp(encoded_X)

print(f"\nNSP测试:")
print(f"输入形状: {encoded_X.shape}")
print(f"NSP预测输出形状: {nsp_Y_hat.shape}")
# 预期: torch.Size([2, 2]) - (batch_size, 2 classes)

# 模拟真实标签：0表示是下一句，1表示不是
nsp_y = torch.tensor([0, 1])

# 计算损失
nsp_l = loss(nsp_Y_hat, nsp_y)

print(f"NSP损失形状: {nsp_l.shape}")
# 预期: torch.Size([2])


#@save
class BERTModel(nn.Module):
    """
    完整的BERT模型
    
    整合了三个部分：
    1. BERTEncoder: 双向Transformer编码器
    2. MaskLM: 掩蔽语言模型任务头
    3. NextSentencePred: 下一句预测任务头
    
    预训练时同时使用MLM和NSP两个任务
    微调时通常只使用Encoder的输出，任务头根据下游任务替换
    """
    
    def __init__(self, vocab_size, num_hiddens, norm_shape, ffn_num_input,
                 ffn_num_hiddens, num_heads, num_layers, dropout,
                 max_len=1000, key_size=768, query_size=768, value_size=768,
                 hid_in_features=768, mlm_in_features=768,
                 nsp_in_features=768):
        """
        参数:
            vocab_size, num_hiddens等: BERTEncoder的参数
            hid_in_features: 用于NSP任务的隐藏层输入维度
            mlm_in_features: MLM任务的输入维度
            nsp_in_features: NSP任务的输入维度
        """
        super(BERTModel, self).__init__()
        
        # ===== BERT编码器 =====
        self.encoder = BERTEncoder(vocab_size, num_hiddens, norm_shape,
                    ffn_num_input, ffn_num_hiddens, num_heads, num_layers,
                    dropout, max_len=max_len, key_size=key_size,
                    query_size=query_size, value_size=value_size)
        
        # ===== 用于NSP的隐藏层 =====
        # 将<cls>标记的表示映射到num_hiddens维度，然后通过tanh激活
        # 这是BERT原始论文的设计，虽然简单线性层也够用
        self.hidden = nn.Sequential(
            nn.Linear(hid_in_features, num_hiddens),
            nn.Tanh()
        )
        
        # ===== MLM任务头 =====
        self.mlm = MaskLM(vocab_size, num_hiddens, mlm_in_features)
        
        # ===== NSP任务头 =====
        self.nsp = NextSentencePred(nsp_in_features)

    def forward(self, tokens, segments, valid_lens=None,
                pred_positions=None):
        """
        前向传播
        
        参数:
            tokens: 词元ID，形状 (batch_size, seq_len)
            segments: 片段索引，形状 (batch_size, seq_len)
            valid_lens: 有效长度，形状 (batch_size,)
            pred_positions: MLM预测位置，形状 (batch_size, num_mlm_preds)
                          如果为None，则不进行MLM预测
        
        返回:
            encoded_X: 编码器输出，形状 (batch_size, seq_len, num_hiddens)
            mlm_Y_hat: MLM预测结果，形状 (batch_size, num_mlm_preds, vocab_size)
                      如果pred_positions为None，则为None
            nsp_Y_hat: NSP预测结果，形状 (batch_size, 2)
        """
        # 通过BERT编码器获取上下文表示
        encoded_X = self.encoder(tokens, segments, valid_lens)
        
        # ===== MLM预测 =====
        if pred_positions is not None:
            # 如果有指定预测位置，进行MLM预测
            mlm_Y_hat = self.mlm(encoded_X, pred_positions)
        else:
            # 微调阶段通常不需要MLM
            mlm_Y_hat = None
        
        # ===== NSP预测 =====
        # 使用<cls>标记（索引0）的表示进行分类
        # self.hidden对<cls>标记的表示进行变换
        # 然后通过nsp分类器得到二分类结果
        nsp_Y_hat = self.nsp(self.hidden(encoded_X[:, 0, :]))
        
        return encoded_X, mlm_Y_hat, nsp_Y_hat

In [ ]:
# =============================================================================
# BERT预训练数据预处理 - WikiText-2数据集
# =============================================================================
# 本模块实现BERT预训练所需的数据处理流程，包括：
#   1. 读取WikiText-2语料库
#   2. 生成NSP（下一句预测）任务数据
#   3. 生成MLM（掩蔽语言模型）任务数据
#   4. 数据填充和批次处理
#
# 数据格式：
#   输入：[<cls>] 句子A [<sep>] 句子B [<sep>]
#   MLM标签：被掩蔽位置的原词
#   NSP标签：1（是下一句）或 0（不是下一句）
# =============================================================================

import os
import random
import torch
from d2l import torch as d2l
import time
import logging
from tqdm import tqdm
from datetime import datetime

# 配置日志系统，用于跟踪数据处理的进度和状态
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("bert_data_processing.log"),  # 写入日志文件
        logging.StreamHandler()                            # 输出到控制台
    ]
)
logger = logging.getLogger(__name__)

print("\n" + "="*80)
print(f"BERT 数据处理程序启动 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# 指定本地数据路径（WikiText-2数据集）
data_dir = '../data/wikitext-2'  # 修改为你的实际路径
print(f"[配置] 使用本地数据集路径: {data_dir}")


#@save
def _read_wiki(data_dir):
    """
    读取WikiText数据集文件
    
    WikiText格式：每行是一个段落，段落内句子用' . '分隔
    
    处理流程：
    1. 读取文件所有行
    2. 按' . '分割句子（过滤短段落）
    3. 转小写
    4. 随机打乱段落顺序
    
    参数:
        data_dir: 数据集目录路径
    
    返回:
        paragraphs: 段落列表，每个段落是句子列表
                   [['sentence1', 'sentence2'], ['sentence3', 'sentence4'], ...]
    """
    file_name = os.path.join(data_dir, 'wiki.train.tokens')
    logger.info(f"[数据加载] 正在读取文件: {file_name}")
    
    start_time = time.time()
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        logger.info(f"[数据加载] 成功读取 {len(lines)} 行数据")
    except Exception as e:
        logger.error(f"[错误] 文件读取失败: {e}")
        raise
    
    # 大写字母转换为小写字母，并按句子分割
    paragraphs = []
    skipped_lines = 0
    
    logger.info("[数据处理] 开始段落分割...")
    for line in tqdm(lines, desc="处理行数据"):
        # 只保留包含至少2个句子的行
        if len(line.split(' . ')) >= 2:
            # 去除首尾空白，转小写，按' . '分割
            processed_line = line.strip().lower().split(' . ')
            paragraphs.append(processed_line)
        else:
            skipped_lines += 1
    
    # 随机打乱段落顺序（NSP任务需要随机采样负样本）
    logger.info(f"[数据处理] 加载段落数量: {len(paragraphs)}, 跳过行数: {skipped_lines}")
    random.shuffle(paragraphs)
    
    end_time = time.time()
    logger.info(f"[计时] 数据读取和预处理耗时: {end_time - start_time:.2f}秒")
    return paragraphs


#@save
def _get_next_sentence(sentence, next_sentence, paragraphs):
    """
    生成下一个句子预测任务的数据
    
    NSP任务数据生成策略：
    - 50%概率：返回真实的下一句（正样本，is_next=True）
    - 50%概率：从随机段落中随机选择一句作为"下一句"（负样本，is_next=False）
    
    参数:
        sentence: 当前句子
        next_sentence: 真实的下一句
        paragraphs: 所有段落列表（用于随机采样负样本）
    
    返回:
        sentence: 句子A
        next_sentence: 句子B（可能是真实的，也可能是随机的）
        is_next: 是否为真实的下一句（True/False）
    """
    if random.random() < 0.5:
        # 50%概率：正样本
        is_next = True
    else:
        # 50%概率：负样本，从随机段落中随机选一句
        # random.choice(paragraphs) 随机选一个段落
        # random.choice(...) 再随机选该段落中的一句
        next_sentence = random.choice(random.choice(paragraphs))
        is_next = False
    return sentence, next_sentence, is_next


#@save
def _get_nsp_data_from_paragraph(paragraph, paragraphs, vocab, max_len):
    """
    从单个段落生成NSP任务数据
    
    处理逻辑：
    - 段落内相邻句子组成句子对
    - 对每个句子对调用_get_next_sentence生成正负样本
    - 过滤超过max_len的样本
    
    参数:
        paragraph: 当前段落（句子列表）
        paragraphs: 所有段落列表（用于采样负样本）
        vocab: 词汇表
        max_len: 最大序列长度（包括<cls>和两个<sep>）
    
    返回:
        nsp_data_from_paragraph: 列表，每个元素是(tokens, segments, is_next)
    """
    nsp_data_from_paragraph = []
    
    # 遍历段落中的相邻句子对
    for i in range(len(paragraph) - 1):
        # 生成NSP样本（可能是正样本或负样本）
        tokens_a, tokens_b, is_next = _get_next_sentence(
            paragraph[i], paragraph[i + 1], paragraphs)
        
        # 检查长度限制（考虑'<cls>'和两个'<sep>'词元，共3个）
        if len(tokens_a) + len(tokens_b) + 3 > max_len:
            continue  # 跳过过长的样本
        
        # 构建BERT输入格式并获取segments
        tokens, segments = d2l.get_tokens_and_segments(tokens_a, tokens_b)
        
        nsp_data_from_paragraph.append((tokens, segments, is_next))
    
    return nsp_data_from_paragraph


#@save
def _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds,
                        vocab):
    """
    替换遮蔽语言模型任务中的词元
    
    BERT的MLM掩码策略（对选中的15%词元）：
    - 80%概率：替换为<mask>标记
    - 10%概率：保持原词不变
    - 10%概率：替换为随机词
    
    为什么这样设计？
    - 80%<mask>：让模型学习从上下文推断被掩蔽词
    - 10%保持原词：让模型知道不是所有位置都被掩蔽
    - 10%随机词：让模型不依赖于知道哪个位置被掩蔽
    
    参数:
        tokens: 原始词元列表
        candidate_pred_positions: 候选预测位置（非特殊标记的位置）
        num_mlm_preds: 需要掩蔽的词元数量
        vocab: 词汇表
    
    返回:
        mlm_input_tokens: 掩蔽后的输入词元
        pred_positions_and_labels: 被掩蔽位置及其原词的列表
    """
    # 复制词元列表，避免修改原始数据
    mlm_input_tokens = [token for token in tokens]
    pred_positions_and_labels = []
    
    # 打乱候选位置，确保随机选择
    random.shuffle(candidate_pred_positions)
    
    for mlm_pred_position in candidate_pred_positions:
        # 达到需要掩蔽的数量就停止
        if len(pred_positions_and_labels) >= num_mlm_preds:
            break
        
        # 80%: <mask>, 10%: 保持原词, 10%: 随机词
        rand_val = random.random()
        if rand_val < 0.8:
            # 80%概率替换为<mask>
            masked_token = '<mask>'
        elif rand_val < 0.9:
            # 10%概率保持原词
            masked_token = tokens[mlm_pred_position]
        else:
            # 10%概率替换为随机词
            masked_token = random.choice(vocab.idx_to_token)
        
        # 记录被掩蔽的位置和原词
        mlm_input_tokens[mlm_pred_position] = masked_token
        pred_positions_and_labels.append(
            (mlm_pred_position, tokens[mlm_pred_position]))
    
    return mlm_input_tokens, pred_positions_and_labels


#@save
def _get_mlm_data_from_tokens(tokens, vocab):
    """
    为MLM任务生成数据
    
    处理流程：
    1. 收集非特殊词元的位置（排除<cls>, <sep>）
    2. 随机选择约15%的位置进行掩蔽
    3. 应用BERT的掩码策略
    
    参数:
        tokens: 词元列表
        vocab: 词汇表
    
    返回:
        vocab[mlm_input_tokens]: 掩蔽后的词元ID列表
        pred_positions: 被掩蔽的位置列表
        vocab[mlm_pred_labels]: 被掩蔽位置的原词ID列表
    """
    candidate_pred_positions = []
    
    # 收集非特殊词元的位置（<cls>和<sep>不参与掩蔽）
    for i, token in enumerate(tokens):
        if token not in ['<cls>', '<sep>']:
            candidate_pred_positions.append(i)
    
    # 确定要遮蔽的词元数量（约15%，至少1个）
    num_mlm_preds = max(1, round(len(tokens) * 0.15))
    
    # 应用掩码策略
    mlm_input_tokens, pred_positions_and_labels = _replace_mlm_tokens(
        tokens, candidate_pred_positions, num_mlm_preds, vocab)
    
    # 按位置排序（保持原始顺序）
    pred_positions_and_labels = sorted(pred_positions_and_labels, key=lambda x: x[0])
    pred_positions = [v[0] for v in pred_positions_and_labels]
    mlm_pred_labels = [v[1] for v in pred_positions_and_labels]
    
    # 转换为词汇表ID
    return vocab[mlm_input_tokens], pred_positions, vocab[mlm_pred_labels]


#@save
def _pad_bert_inputs(examples, max_len, vocab):
    """
    填充BERT输入数据，使其具有相同长度
    
    需要填充的字段：
    - token_ids: 词元ID（用<pad>填充）
    - segments: 片段索引（用0填充）
    - valid_lens: 有效长度
    - pred_positions: 预测位置（用0填充）
    - mlm_weights: MLM权重（实际位置为1.0，填充位置为0.0）
    - mlm_labels: MLM标签（用0填充）
    - nsp_labels: NSP标签（无需填充）
    
    参数:
        examples: 样本列表，每个样本是(token_ids, pred_positions, mlm_pred_label_ids, segments, is_next)
        max_len: 最大序列长度
        vocab: 词汇表
    
    返回:
        填充后的各字段张量元组
    """
    max_num_mlm_preds = round(max_len * 0.15)  # 最大MLM预测数量
    print(f"[数据填充] 开始填充数据，最大长度: {max_len}, 最大MLM预测数: {max_num_mlm_preds}")
    
    all_token_ids, all_segments, valid_lens = [], [], []
    all_pred_positions, all_mlm_weights, all_mlm_labels = [], [], []
    nsp_labels = []
    
    start_time = time.time()
    for example in examples:
        (token_ids, pred_positions, mlm_pred_label_ids, segments, is_next) = example
        
        # ===== 填充token_ids =====
        pad_len = max_len - len(token_ids)
        all_token_ids.append(torch.tensor(token_ids + [vocab['<pad>']] * pad_len, dtype=torch.long))
        
        # ===== 填充segments =====
        all_segments.append(torch.tensor(segments + [0] * pad_len, dtype=torch.long))
        
        # ===== 记录有效长度 =====
        valid_lens.append(torch.tensor(len(token_ids), dtype=torch.float32))
        
        # ===== 填充预测位置 =====
        pad_mlm_len = max_num_mlm_preds - len(pred_positions)
        all_pred_positions.append(torch.tensor(pred_positions + [0] * pad_mlm_len, dtype=torch.long))
        
        # ===== MLM权重（用于加权平均损失）=====
        # 实际预测位置为1.0，填充位置为0.0
        all_mlm_weights.append(
            torch.tensor([1.0] * len(mlm_pred_label_ids) + [0.0] * pad_mlm_len,
            dtype=torch.float32))
        
        # ===== 填充MLM标签 =====
        all_mlm_labels.append(torch.tensor(mlm_pred_label_ids + [0] * pad_mlm_len, dtype=torch.long))
        
        # ===== NSP标签 =====
        nsp_labels.append(torch.tensor(is_next, dtype=torch.long))
    
    end_time = time.time()
    print(f"[计时] 数据填充耗时: {end_time - start_time:.2f}秒")
    print(f"[数据统计] 总样本数: {len(all_token_ids)}")
    
    return (all_token_ids, all_segments, valid_lens, all_pred_positions,
            all_mlm_weights, all_mlm_labels, nsp_labels)


#@save
class _WikiTextDataset(torch.utils.data.Dataset):
    """
    WikiText数据集类，用于BERT预训练
    
    数据集构建流程：
    1. 读取段落并分词
    2. 构建词汇表
    3. 生成NSP样本
    4. 生成MLM样本
    5. 填充数据
    """
    
    def __init__(self, paragraphs, max_len):
        """
        初始化数据集
        
        参数:
            paragraphs: 段落列表
            max_len: 最大序列长度
        """
        logger.info(f"[数据集] 开始构建数据集，最大序列长度: {max_len}")
        start_time = time.time()
        self.construction_time = start_time
        self.last_log_time = start_time
        
        # 进度跟踪
        self.progress = {
            'paragraphs_processed': 0,
            'nsp_samples_generated': 0,
            'mlm_samples_processed': 0
        }
        
        # 设置进度更新间隔（秒）
        self.progress_interval = 30
        
        # ===== 分词处理 =====
        logger.info("[分词] 正在进行段落分词...")
        tokenized_paragraphs = []
        for i, paragraph in enumerate(tqdm(paragraphs, desc="分词段落")):
            # 对每个句子的每个词进行分词
            tokenized_paragraphs.append(d2l.tokenize(paragraph, token='word'))
            self._log_progress(i, len(paragraphs), "段落分词")
        
        paragraphs = tokenized_paragraphs
        
        # ===== 创建词汇表 =====
        logger.info("[词汇表] 正在构建词汇表...")
        # 展平所有句子用于构建词汇表
        sentences = [sentence for paragraph in paragraphs for sentence in paragraph]
        self.vocab = d2l.Vocab(sentences, min_freq=5, reserved_tokens=[
            '<pad>', '<mask>', '<cls>', '<sep>'])
        logger.info(f"[词汇表] 词汇表大小: {len(self.vocab)}")
        
        # ===== 生成NSP任务数据 =====
        logger.info("[NSP任务] 正在生成下一句预测数据...")
        examples = []
        for i, paragraph in enumerate(tqdm(paragraphs, desc="生成NSP数据")):
            nsp_data = _get_nsp_data_from_paragraph(paragraph, paragraphs, self.vocab, max_len)
            examples.extend(nsp_data)
            self.progress['nsp_samples_generated'] = len(examples)
            self.progress['paragraphs_processed'] = i + 1
            self._log_progress(i, len(paragraphs), "生成NSP样本")
        
        logger.info(f"[NSP任务] 共生成 {len(examples)} 个NSP样本")
        
        # ===== 生成MLM任务数据 =====
        logger.info("[MLM任务] 正在生成遮蔽语言模型数据...")
        mlm_examples = []
        for i, (tokens, segments, is_next) in enumerate(tqdm(examples, desc="生成MLM数据")):
            mlm_data = _get_mlm_data_from_tokens(tokens, self.vocab)
            mlm_examples.append(mlm_data + (segments, is_next))
            self.progress['mlm_samples_processed'] = i + 1
            if i > 0 and i % 10000 == 0:
                logger.info(f"[进度] 已处理 {i+1}/{len(examples)} 个MLM样本")
        
        # ===== 填充数据 =====
        logger.info("[数据填充] 正在进行数据填充...")
        (self.all_token_ids, self.all_segments, self.valid_lens,
         self.all_pred_positions, self.all_mlm_weights,
         self.all_mlm_labels, self.nsp_labels) = _pad_bert_inputs(
            mlm_examples, max_len, self.vocab)
        
        end_time = time.time()
        logger.info(f"[计时] 数据集构建总耗时: {end_time - start_time:.2f}秒")
        logger.info(f"[数据集] 最终数据集大小: {len(self.all_token_ids)} 个样本")
    
    def _log_progress(self, current, total, task_name):
        """记录进度，避免过于频繁的日志输出"""
        current_time = time.time()
        if current_time - self.last_log_time > self.progress_interval:
            elapsed = current_time - self.construction_time
            progress_percent = (current + 1) / total * 100
            remaining_time = (elapsed / (current + 1)) * (total - current - 1) if current > 0 else 0
            
            logger.info(
                f"[进度] {task_name}: {current+1}/{total} ({progress_percent:.1f}%) | "
                f"已用时间: {elapsed:.1f}s | 预计剩余时间: {remaining_time:.1f}s"
            )
            self.last_log_time = current_time
    
    def __getitem__(self, idx):
        """获取单个样本"""
        if idx == 0:
            logger.debug(f"[数据集] 正在访问索引 {idx} 的数据")
        return (self.all_token_ids[idx], self.all_segments[idx],
                self.valid_lens[idx], self.all_pred_positions[idx],
                self.all_mlm_weights[idx], self.all_mlm_labels[idx],
                self.nsp_labels[idx])

    def __len__(self):
        """返回数据集大小"""
        return len(self.all_token_ids)


#@save
def load_data_wiki(batch_size, max_len):
    """
    加载WikiText-2数据集
    
    参数:
        batch_size: 批次大小
        max_len: 最大序列长度
    
    返回:
        train_iter: 数据加载器
        vocab: 词汇表
    """
    logger.info("\n" + "="*80)
    logger.info(f"开始加载数据，批大小: {batch_size}, 最大序列长度: {max_len}")
    
    # 获取数据加载工作进程数（Windows上设为0避免多进程问题）
#     num_workers = d2l.get_dataloader_workers()
    num_workers = 0
    logger.info(f"[配置] 使用 {num_workers} 个工作进程加载数据")
    
    # 读取数据
    paragraphs = _read_wiki(data_dir)
    
    # 创建数据集
    train_set = _WikiTextDataset(paragraphs, max_len)
    
    # 创建数据加载器
    logger.info("[数据加载器] 正在创建数据加载器...")
    train_iter = torch.utils.data.DataLoader(
        train_set, batch_size, shuffle=True, num_workers=num_workers)
    
    logger.info("="*80)
    logger.info("数据加载完成!")
    logger.info("="*80 + "\n")
    
    return train_iter, train_set.vocab


# =============================================================================
# 主程序入口
# =============================================================================
if __name__ == "__main__":
    logger.info("===== 开始主程序 =====")
    batch_size, max_len = 512, 64
    logger.info(f"[参数] 批大小: {batch_size}, 最大序列长度: {max_len}")
    
    # 加载数据
    train_iter, vocab = load_data_wiki(batch_size, max_len)
    
    # 检查第一个批次
    logger.info("\n检查第一个批次的数据形状:")
    start_time = time.time()
    last_log_time = start_time
    batch_found = False
    
    logger.info("[数据加载] 开始从数据加载器获取第一个批次...")
    try:
        for i, batch in enumerate(train_iter):
            if i == 0:  # 只处理第一个批次
                (tokens_X, segments_X, valid_lens_x, pred_positions_X, mlm_weights_X,
                 mlm_Y, nsp_y) = batch
                batch_found = True
                break
            
            # 每10个批次记录一次进度
            current_time = time.time()
            if current_time - last_log_time > 5:  # 每5秒记录一次
                logger.info(f"[进度] 已处理 {i+1} 个批次...")
                last_log_time = current_time
    except Exception as e:
        logger.error(f"[错误] 数据加载失败: {e}")
        raise
    
    if not batch_found:
        logger.error("[错误] 未能获取任何批次数据")
    else:
        end_time = time.time()
        logger.info(f"[计时] 获取第一个批次耗时: {end_time - start_time:.4f}秒")
        logger.info(f"tokens_X 形状: {tokens_X.shape}")        # (batch_size, max_len)
        logger.info(f"segments_X 形状: {segments_X.shape}")    # (batch_size, max_len)
        logger.info(f"valid_lens_x 形状: {valid_lens_x.shape}") # (batch_size,)
        logger.info(f"pred_positions_X 形状: {pred_positions_X.shape}") # (batch_size, max_num_mlm_preds)
        logger.info(f"mlm_weights_X 形状: {mlm_weights_X.shape}") # (batch_size, max_num_mlm_preds)
        logger.info(f"mlm_Y 形状: {mlm_Y.shape}")             # (batch_size, max_num_mlm_preds)
        logger.info(f"nsp_y 形状: {nsp_y.shape}")             # (batch_size,)
    
    # 词汇表信息
    logger.info(f"\n词汇表大小: {len(vocab)}")
    logger.info("特殊词元示例:")
    logger.info(f"<pad>: {vocab['<pad>']}")
    logger.info(f"<mask>: {vocab['<mask>']}")
    logger.info(f"<cls>: {vocab['<cls>']}")
    logger.info(f"<sep>: {vocab['<sep>']}")
    
    logger.info("\n===== 程序执行完成 =====")

# 打印词汇表大小（在Jupyter中显示）
print(len(vocab))

In [ ]:
# =============================================================================
# BERT预训练 - 训练循环实现
# =============================================================================
# 本模块实现BERT的预训练流程，包括：
#   1. 修复数据读取函数（处理编码问题）
#   2. 加载WikiText数据集
#   3. 定义损失计算函数（MLM + NSP）
#   4. 实现训练循环
#   5. 可视化训练过程
# =============================================================================

import torch
from torch import nn
from d2l import torch as d2l

# 覆盖 d2l 库中的 _read_wiki 函数，修复Windows编码问题
def _read_wiki(data_dir):
    """
    修复版WikiText读取函数
    
    原d2l库的函数可能存在编码问题，这里显式使用UTF-8编码
    """
    file_name = os.path.join(data_dir, 'wiki.train.tokens')
    print(f"读取文件: {file_name}")
    
    # 明确使用 UTF-8 编码，避免Windows系统默认编码问题
    with open(file_name, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # 处理逻辑：只保留包含至少2个句子的行
    # 转小写并按' . '分割
    paragraphs = [line.strip().lower().split(' . ') 
                  for line in lines if len(line.split(' . ')) >= 2]
    
    # 随机打乱
    random.shuffle(paragraphs)
    return paragraphs

# 将修复后的函数应用到 d2l 库中
d2l.torch._read_wiki = _read_wiki

# =============================================================================
# 加载数据
# =============================================================================
batch_size, max_len = 512, 64

# 使用d2l提供的函数加载WikiText数据集
train_iter, vocab = d2l.load_data_wiki(batch_size, max_len)

# =============================================================================
# 创建BERT模型
# =============================================================================
# 使用较小的模型配置进行演示（实际预训练会用更大的模型）
net = d2l.BERTModel(
    len(vocab),           # 词汇表大小
    num_hiddens=128,      # 隐藏层维度（实际BERT-base是768）
    norm_shape=[128],     # 层归一化形状
    ffn_num_input=128,    # FFN输入维度
    ffn_num_hiddens=256,  # FFN隐藏层维度
    num_heads=2,          # 注意力头数（实际BERT-base是12）
    num_layers=2,         # Transformer层数（实际BERT-base是12）
    dropout=0.2,          # Dropout概率
    key_size=128,         # Key维度
    query_size=128,       # Query维度
    value_size=128,       # Value维度
    hid_in_features=128,  # NSP隐藏层输入
    mlm_in_features=128,  # MLM输入维度
    nsp_in_features=128   # NSP输入维度
)

# 获取可用设备（优先使用GPU）
devices = d2l.try_all_gpus()

# 定义损失函数（用于MLM和NSP）
loss = nn.CrossEntropyLoss()


#@save
def _get_batch_loss_bert(net, loss, vocab_size, tokens_X,
                         segments_X, valid_lens_x,
                         pred_positions_X, mlm_weights_X,
                         mlm_Y, nsp_y):
    """
    计算BERT预训练的损失（MLM + NSP）
    
    参数:
        net: BERT模型
        loss: 损失函数（CrossEntropyLoss）
        vocab_size: 词汇表大小
        tokens_X: 输入词元ID，形状 (batch_size, max_len)
        segments_X: 片段索引，形状 (batch_size, max_len)
        valid_lens_x: 有效长度，形状 (batch_size,)
        pred_positions_X: MLM预测位置，形状 (batch_size, max_num_mlm_preds)
        mlm_weights_X: MLM权重，形状 (batch_size, max_num_mlm_preds)
        mlm_Y: MLM标签，形状 (batch_size, max_num_mlm_preds)
        nsp_y: NSP标签，形状 (batch_size,)
    
    返回:
        mlm_l: MLM损失（标量）
        nsp_l: NSP损失（标量）
        l: 总损失（标量）
    
    损失计算说明：
    - MLM损失：对预测位置的损失加权平均
    - NSP损失：二分类交叉熵
    - 总损失 = MLM损失 + NSP损失
    """
    # 前向传播
    # encoded_X: (batch_size, max_len, num_hiddens)
    # mlm_Y_hat: (batch_size, max_num_mlm_preds, vocab_size)
    # nsp_Y_hat: (batch_size, 2)
    _, mlm_Y_hat, nsp_Y_hat = net(tokens_X, segments_X,
                                  valid_lens_x.reshape(-1),
                                  pred_positions_X)
    
    # ===== 计算MLM损失 =====
    # 将预测结果展平：(batch_size * max_num_mlm_preds, vocab_size)
    # 将标签展平：(batch_size * max_num_mlm_preds,)
    mlm_l = loss(mlm_Y_hat.reshape(-1, vocab_size), mlm_Y.reshape(-1))
    
    # 乘以权重并求和，然后除以权重和（加权平均）
    # mlm_weights_X.reshape(-1, 1): (batch_size * max_num_mlm_preds, 1)
    mlm_l = mlm_l * mlm_weights_X.reshape(-1, 1)
    mlm_l = mlm_l.sum() / (mlm_weights_X.sum() + 1e-8)  # 加epsilon避免除0
    
    # ===== 计算NSP损失 =====
    # 标准的二分类交叉熵
    nsp_l = loss(nsp_Y_hat, nsp_y)
    
    # ===== 总损失 =====
    l = mlm_l + nsp_l
    
    return mlm_l, nsp_l, l


def train_bert(train_iter, net, loss, vocab_size, devices, num_steps):
    """
    BERT预训练函数
    
    参数:
        train_iter: 数据迭代器
        net: BERT模型
        loss: 损失函数
        vocab_size: 词汇表大小
        devices: 设备列表
        num_steps: 训练步数
    
    训练流程：
    1. 将模型包装为DataParallel（多GPU）
    2. 定义Adam优化器
    3. 迭代训练，计算MLM和NSP损失
    4. 可视化损失曲线
    """
    # 使用DataParallel支持多GPU训练
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    
    # Adam优化器（BERT使用Adam with weight decay）
    trainer = torch.optim.Adam(net.parameters(), lr=0.01)
    
    # 计时器和可视化
    step, timer = 0, d2l.Timer()
    animator = d2l.Animator(xlabel='step', ylabel='loss',
                            xlim=[1, num_steps], 
                            legend=['mlm', 'nsp'])
    
    # 累积器：MLM损失和、NSP损失和、样本数、批次计数
    metric = d2l.Accumulator(4)
    num_steps_reached = False
    
    # ===== 训练循环 =====
    while step < num_steps and not num_steps_reached:
        for tokens_X, segments_X, valid_lens_x, pred_positions_X,\
            mlm_weights_X, mlm_Y, nsp_y in train_iter:
            
            # 将数据移动到设备
            tokens_X = tokens_X.to(devices[0])
            segments_X = segments_X.to(devices[0])
            valid_lens_x = valid_lens_x.to(devices[0])
            pred_positions_X = pred_positions_X.to(devices[0])
            mlm_weights_X = mlm_weights_X.to(devices[0])
            mlm_Y, nsp_y = mlm_Y.to(devices[0]), nsp_y.to(devices[0])
            
            # 梯度清零
            trainer.zero_grad()
            
            # 计时开始
            timer.start()
            
            # 计算损失
            mlm_l, nsp_l, l = _get_batch_loss_bert(
                net, loss, vocab_size, tokens_X, segments_X, valid_lens_x,
                pred_positions_X, mlm_weights_X, mlm_Y, nsp_y)
            
            # 反向传播
            l.backward()
            
            # 参数更新
            trainer.step()
            
            # 累积统计
            metric.add(mlm_l, nsp_l, tokens_X.shape[0], 1)
            
            # 计时结束
            timer.stop()
            
            # 可视化更新
            animator.add(step + 1,
                         (metric[0] / metric[3], metric[1] / metric[3]))
            
            step += 1
            
            # 达到指定步数就停止
            if step == num_steps:
                num_steps_reached = True
                break

    # 打印训练结果
    print(f'MLM loss {metric[0] / metric[3]:.3f}, '
          f'NSP loss {metric[1] / metric[3]:.3f}')
    print(f'{metric[2] / timer.sum():.1f} sentence pairs/sec on '
          f'{str(devices)}')


# =============================================================================
# 执行训练
# =============================================================================
# 训练50步用于演示
train_bert(train_iter, net, loss, len(vocab), devices, 50)

In [3]:
import d2l
print(d2l.__file__)

D:\anaconda3\envs\first\lib\site-packages\d2l\__init__.py


In [ ]:
# =============================================================================
# BERT词向量获取与语义理解测试
# =============================================================================
# 本模块演示如何使用预训练的BERT模型获取词向量表示
# 关键概念：同一个词在不同上下文中会有不同的表示（上下文相关词向量）
#
# 示例：
#   "crane"在"a crane is flying"中指"鹤"
#   "crane"在"a crane driver came"中指"起重机"
# BERT能根据上下文给出不同的向量表示
# =============================================================================

import d2l


def get_bert_encoding(net, tokens_a, tokens_b=None):
    """
    获取BERT编码表示
    
    参数:
        net: BERT模型
        tokens_a: 第一个句子的词元列表
        tokens_b: 第二个句子的词元列表（可选）
    
    返回:
        encoded_X: BERT编码输出，形状 (1, seq_len, num_hiddens)
    
    说明：
    - [CLS]标记（索引0）的输出通常用于分类任务
    - 其他位置的输出是对应词元的上下文相关表示
    """
    # 构建输入序列和片段索引
    tokens, segments = d2l.get_tokens_and_segments(tokens_a, tokens_b)
    
    # 转换为词汇表ID并添加批次维度
    token_ids = torch.tensor(vocab[tokens], device=devices[0]).unsqueeze(0)
    # unsqueeze(0): (seq_len,) -> (1, seq_len)
    
    # 转换为张量并添加批次维度
    segments = torch.tensor(segments, device=devices[0]).unsqueeze(0)
    
    # 有效长度
    valid_len = torch.tensor(len(tokens), device=devices[0]).unsqueeze(0)
    
    # 前向传播（不需要MLM和NSP预测）
    encoded_X, _, _ = net(token_ids, segments, valid_len)
    
    return encoded_X


# =============================================================================
# 测试1：单句编码
# =============================================================================
tokens_a = ['a', 'crane', 'is', 'flying']

# 获取BERT编码
encoded_text = get_bert_encoding(net, tokens_a)

# 词元序列：[<cls>,'a','crane','is','flying','<sep>]
# 索引：    [  0     1     2      3      4       5   ]

# 提取<cls>标记的表示（用于句子级别的分类任务）
encoded_text_cls = encoded_text[:, 0, :]

# 提取"crane"（索引2）的表示
encoded_text_crane = encoded_text[:, 2, :]

print("=" * 60)
print("测试1：单句编码")
print("=" * 60)
print(f"输入词元: {['<cls>'] + tokens_a + ['<sep>']}")
print(f"编码输出形状: {encoded_text.shape}")
print(f"  - batch_size: {encoded_text.shape[0]}")
print(f"  - seq_len: {encoded_text.shape[1]}")
print(f"  - hidden_size: {encoded_text.shape[2]}")
print(f"\n[CLS]标记表示形状: {encoded_text_cls.shape}")
print(f"'crane'表示形状: {encoded_text_crane.shape}")
print(f"'crane'表示前3维: {encoded_text_crane[0][:3]}")


# =============================================================================
# 测试2：句子对编码（NSP格式）
# =============================================================================
tokens_a, tokens_b = ['a', 'crane', 'driver', 'came'], ['he', 'just', 'left']

# 获取BERT编码
encoded_pair = get_bert_encoding(net, tokens_a, tokens_b)

# 词元序列：[<cls>,'a','crane','driver','came','<sep>','he','just','left','<sep>]
# 索引：    [  0     1     2       3        4      5      6     7      8      9   ]

# 提取<cls>标记的表示
encoded_pair_cls = encoded_pair[:, 0, :]

# 提取"crane"（索引2）的表示
encoded_pair_crane = encoded_pair[:, 2, :]

print("\n" + "=" * 60)
print("测试2：句子对编码")
print("=" * 60)
print(f"句子A: {tokens_a}")
print(f"句子B: {tokens_b}")
print(f"输入词元: {['<cls>'] + tokens_a + ['<sep>'] + tokens_b + ['<sep>']}")
print(f"编码输出形状: {encoded_pair.shape}")
print(f"\n[CLS]标记表示形状: {encoded_pair_cls.shape}")
print(f"'crane'表示形状: {encoded_pair_crane.shape}")
print(f"'crane'表示前3维: {encoded_pair_crane[0][:3]}")


# =============================================================================
# 结果分析
# =============================================================================
print("\n" + "=" * 60)
print("结果分析")
print("=" * 60)
print("""
关键观察：
1. 同一个词"crane"在不同上下文中得到不同表示
   - 单句中（"a crane is flying"）：表示"鹤"
   - 句子对中（"a crane driver came"）：表示"起重机"

2. [CLS]标记的表示捕获了整个序列的语义信息
   - 常用于下游分类任务（如情感分析、文本分类）

3. BERT的词向量是上下文相关的（Contextualized）
   - 与传统的静态词向量（如Word2Vec）不同
   - 同一个词在不同句子中会有不同表示
""")

# 返回编码结果供进一步分析
encoded_text.shape, encoded_text_cls.shape, encoded_text_crane[0][:3]